In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.8 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [5]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"
EVALUATION_PATH = PROJECT_PATH / "evaluation"

In [6]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=20
):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "faiss_rank": rank,
            "faiss_score": float(
                scores[0][rank - 1]
            ),
            "index": int(idx),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [8]:
query = (
    "How can uterine EMG be used "
    "to predict preterm labor?"
)

candidates = retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=20
)

In [9]:
for result in candidates:

    print(
        f"\nFAISS Rank: {result['faiss_rank']}"
    )

    print(
        f"Score: {result['faiss_score']:.4f}"
    )

    print(
        f"Paper: {result['paper']}"
    )

    print(
        f"Chunk: {result['chunk_id']}"
    )

    print(
        result["text"][:300]
    )


FAISS Rank: 1
Score: 0.7983
Paper: paper7
Chunk: 35
that lead to delivery are reflected in changes in several EHG parameters (Lucovnik et al 2011). Several stud-
ies have shown that EHG features are ‘dynamic’ and change throughout pregnancy (Devedeux et al 1993). At 
early gestational ages uterine electrical activity is scarce and poorly coordinated,

FAISS Rank: 2
Score: 0.7676
Paper: paper1
Chunk: 194
fication of uterine EMG signals using supervised classification method. Biomed 
Sci Eng 2010;3(9):837–42. http://dx.doi.org/10.4236/jbise.2010.39113.
[42] Verdenik Ivan, Pajntar Marjan, Leskosek Brane. Uterine electrical activity 
as predictor of preterm birth in women with preterm contractions. Eur

FAISS Rank: 3
Score: 0.7555
Paper: paper4
Chunk: 63
no. 12, pp. 1182–1187, 1986. 
[20] G. Fele-ˇZorˇz, G. Kavˇsek, ˇZ. Novak-Antoliˇc, F. Jager, A comparison of various linear 
and non-linear signal processing techniques to separate uterine EMG records of 
term and pre-term delivery groups

In [10]:
!pip install sentence-transformers

In [11]:
from sentence_transformers import CrossEncoder

In [12]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [13]:
pairs = [
    [query, result["text"]]
    for result in candidates
]

In [14]:
len(pairs)

20

In [15]:
rerank_scores = reranker.predict(
    pairs
)

In [16]:
for result, score in zip(
    candidates,
    rerank_scores
):

    result["rerank_score"] = float(score)

In [17]:
reranked_results = sorted(
    candidates,
    key=lambda x: x["rerank_score"],
    reverse=True
)

In [18]:
top_results = reranked_results[:5]

In [19]:
for rank, result in enumerate(
    top_results,
    start=1
):

    print(
        f"\nRerank Rank: {rank}"
    )

    print(
        f"Rerank Score: "
        f"{result['rerank_score']:.4f}"
    )

    print(
        f"Original FAISS Rank: "
        f"{result['faiss_rank']}"
    )

    print(
        f"Paper: {result['paper']}"
    )

    print(
        f"Chunk: {result['chunk_id']}"
    )

    print(
        result["text"][:500]
    )


Rerank Rank: 1
Rerank Score: 5.8180
Original FAISS Rank: 10
Paper: paper2
Chunk: 17
that uterine contractions are accompanied by reproducible electrical signals detectable
on the abdominal surface. In 1993, Devedeux and colleagues published a critical review
synthesizing animal and human uterine EMG findings, solidifying electrohysterography as
a viable modality for monitoring pregnancy and labor [10]. Since then, numerous studies
have advanced EHG technology and analysis, aiming to develop it into a clinical tool for
labor monitoring and preterm-birth prediction.
2.2. Physiolo

Rerank Rank: 2
Rerank Score: 5.2657
Original FAISS Rank: 1
Paper: paper7
Chunk: 35
that lead to delivery are reflected in changes in several EHG parameters (Lucovnik et al 2011). Several stud-
ies have shown that EHG features are ‘dynamic’ and change throughout pregnancy (Devedeux et al 1993). At 
early gestational ages uterine electrical activity is scarce and poorly coordinated, however as labor approaches 


In [20]:
def retrieve_and_rerank(
    query,
    embedding_model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
):

    # Stage 1: FAISS retrieval
    candidates = retrieve_candidates(
        query,
        embedding_model,
        index,
        metadata,
        k=candidate_k
    )

    # Stage 2: Cross-Encoder reranking
    pairs = [
        [query, result["text"]]
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    for result, score in zip(
        candidates,
        scores
    ):
        result["rerank_score"] = float(score)

    # Sort
    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidates[:final_k]

In [21]:
results = retrieve_and_rerank(
    query,
    model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
)

In [22]:
faiss_results = retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=5
)

reranked_results = retrieve_and_rerank(
    query,
    model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
)

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

In [25]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"
EVALUATION_PATH = PROJECT_PATH / "evaluation"

In [26]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [27]:
from pathlib import Path


def normalize_paper_name(name):
    return Path(str(name)).stem.lower().strip()

In [28]:
def precision_at_k(results, relevant_papers, k=5):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    retrieved = results[:k]

    relevant_count = sum(
        normalize_paper_name(r["paper"])
        in relevant_papers
        for r in retrieved
    )

    return relevant_count / k

In [29]:
def recall_at_k(results, relevant_papers, k=5):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    retrieved_papers = {
        normalize_paper_name(r["paper"])
        for r in results[:k]
    }

    relevant_retrieved = (
        retrieved_papers.intersection(
            relevant_papers
        )
    )

    return (
        len(relevant_retrieved)
        / len(relevant_papers)
    )

In [55]:
def reciprocal_rank(results, relevant_papers):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    for rank, result in enumerate(results, start=1):

        if (
            normalize_paper_name(result["paper"])
            in relevant_papers
        ):
            return 1 / rank

    return 0.0

In [56]:
evaluation_queries = [
    {
        "query": "What are the characteristics of uterine EMG signals?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    },
    {
        "query": "How can uterine EMG predict preterm labor?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    },
    {
        "query": "What signal processing methods are used for uterine EMG?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    }
]

In [57]:
query = evaluation_queries[0]["query"]

relevant_papers = (
    evaluation_queries[0]["relevant_papers"]
)

In [58]:
faiss_results = retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=5
)

In [59]:
reranked_results = retrieve_and_rerank(
    query,
    model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
)

In [60]:
faiss_precision = precision_at_k(
    faiss_results,
    relevant_papers,
    k=5
)

faiss_recall = recall_at_k(
    faiss_results,
    relevant_papers,
    k=5
)

faiss_mrr = reciprocal_rank(
    faiss_results,
    relevant_papers
)

In [61]:
print("FAISS")
print("Precision@5:", faiss_precision)
print("Recall@5:", faiss_recall)
print("MRR:", faiss_mrr)

FAISS
Precision@5: 1.0
Recall@5: 0.2
MRR: 1.0


In [62]:
rerank_precision = precision_at_k(
    reranked_results,
    relevant_papers,
    k=5
)

rerank_recall = recall_at_k(
    reranked_results,
    relevant_papers,
    k=5
)

rerank_mrr = reciprocal_rank(
    reranked_results,
    relevant_papers
)

In [63]:
print("FAISS + Reranker")
print("Precision@5:", rerank_precision)
print("Recall@5:", rerank_recall)
print("MRR:", rerank_mrr)

FAISS + Reranker
Precision@5: 1.0
Recall@5: 0.2
MRR: 1.0


In [64]:
comparison = pd.DataFrame([
    {
        "Method": "FAISS",
        "Precision@5": faiss_precision,
        "Recall@5": faiss_recall,
        "MRR": faiss_mrr
    },
    {
        "Method": "FAISS + Reranker",
        "Precision@5": rerank_precision,
        "Recall@5": rerank_recall,
        "MRR": rerank_mrr
    }
])

comparison

,Method,Precision@5,Recall@5,MRR
0,FAISS,1.0,0.2,1.0
1,FAISS + Reranker,1.0,0.2,1.0


In [65]:
comparison_results = []

In [66]:
for item in evaluation_queries:

    query = item["query"]
    relevant_papers = item["relevant_papers"]

    # --------------------------------
    # 1. FAISS ONLY
    # --------------------------------

    faiss_results = retrieve_candidates(
        query,
        model,
        index,
        metadata,
        k=5
    )

    faiss_precision = precision_at_k(
        faiss_results,
        relevant_papers,
        k=5
    )

    faiss_recall = recall_at_k(
        faiss_results,
        relevant_papers,
        k=5
    )

    faiss_mrr = reciprocal_rank(
        faiss_results,
        relevant_papers
    )


    # --------------------------------
    # 2. FAISS + RERANKER
    # --------------------------------

    reranked_results = retrieve_and_rerank(
        query,
        model,
        index,
        metadata,
        reranker,
        candidate_k=20,
        final_k=5
    )

    rerank_precision = precision_at_k(
        reranked_results,
        relevant_papers,
        k=5
    )

    rerank_recall = recall_at_k(
        reranked_results,
        relevant_papers,
        k=5
    )

    rerank_mrr = reciprocal_rank(
        reranked_results,
        relevant_papers
    )


    # --------------------------------
    # 3. SAVE RESULTS
    # --------------------------------

    comparison_results.append({

        "query": query,

        "FAISS_Precision@5":
            faiss_precision,

        "FAISS_Recall@5":
            faiss_recall,

        "FAISS_MRR":
            faiss_mrr,

        "Reranker_Precision@5":
            rerank_precision,

        "Reranker_Recall@5":
            rerank_recall,

        "Reranker_MRR":
            rerank_mrr
    })

In [67]:
comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,query,FAISS_Precision@5,FAISS_Recall@5,FAISS_MRR,Reranker_Precision@5,Reranker_Recall@5,Reranker_MRR
0,What are the characteristics of uterine EMG si...,1.0,0.2,1.000000,1.0,0.2,1.0
1,How can uterine EMG predict preterm labor?,0.4,0.4,0.333333,0.4,0.4,1.0
2,What signal processing methods are used for ut...,1.0,0.2,1.000000,1.0,0.4,1.0


In [68]:
summary = pd.DataFrame({
    "Metric": [
        "Precision@5",
        "Recall@5",
        "MRR"
    ],

    "FAISS": [
        comparison_df[
            "FAISS_Precision@5"
        ].mean(),

        comparison_df[
            "FAISS_Recall@5"
        ].mean(),

        comparison_df[
            "FAISS_MRR"
        ].mean()
    ],

    "FAISS + Reranker": [
        comparison_df[
            "Reranker_Precision@5"
        ].mean(),

        comparison_df[
            "Reranker_Recall@5"
        ].mean(),

        comparison_df[
            "Reranker_MRR"
        ].mean()
    ]
})

summary

,Metric,FAISS,FAISS + Reranker
0,Precision@5,0.800000,0.800000
1,Recall@5,0.266667,0.333333
2,MRR,0.777778,1.000000


In [69]:
summary["Improvement"] = (
    summary["FAISS + Reranker"]
    - summary["FAISS"]
)

In [70]:
summary["Improvement_%"] = (
    (
        summary["FAISS + Reranker"]
        - summary["FAISS"]
    )
    / summary["FAISS"]
) * 100

In [71]:
comparison_df.to_csv(
    EVALUATION_PATH /
    "retrieval_comparison.csv",
    index=False
)

In [72]:
summary.to_csv(
    EVALUATION_PATH /
    "retrieval_comparison_summary.csv",
    index=False
)

In [73]:
comparison_df["Precision_Improvement"] = (
    comparison_df["Reranker_Precision@5"]
    - comparison_df["FAISS_Precision@5"]
)

In [74]:
comparison_df[
    [
        "query",
        "FAISS_Precision@5",
        "Reranker_Precision@5",
        "Precision_Improvement"
    ]
]

,query,FAISS_Precision@5,Reranker_Precision@5,Precision_Improvement
0,What are the characteristics of uterine EMG si...,1.0,1.0,0.0
1,How can uterine EMG predict preterm labor?,0.4,0.4,0.0
2,What signal processing methods are used for ut...,1.0,1.0,0.0


In [75]:
print("=" * 100)
print("QUERY:", query)
print("=" * 100)

print("\nFAISS RESULTS")
print("-" * 100)

for r in faiss_results:
    print(
        f"Rank {r['faiss_rank']} | "
        f"Score {r['faiss_score']:.4f} | "
        f"Paper {r['paper']} | "
        f"Chunk {r['chunk_id']}"
    )

print("\nRERANKED RESULTS")
print("-" * 100)

for rank, r in enumerate(
    reranked_results,
    start=1
):

    print(
        f"Rank {rank} | "
        f"Rerank Score {r['rerank_score']:.4f} | "
        f"Original FAISS Rank {r['faiss_rank']} | "
        f"Paper {r['paper']} | "
        f"Chunk {r['chunk_id']}"
    )

QUERY: What signal processing methods are used for uterine EMG?

FAISS RESULTS
----------------------------------------------------------------------------------------------------
Rank 1 | Score 0.7736 | Paper paper2 | Chunk 49
Rank 2 | Score 0.7508 | Paper paper2 | Chunk 41
Rank 3 | Score 0.7410 | Paper paper2 | Chunk 52
Rank 4 | Score 0.7383 | Paper paper2 | Chunk 54
Rank 5 | Score 0.7347 | Paper paper2 | Chunk 22

RERANKED RESULTS
----------------------------------------------------------------------------------------------------
Rank 1 | Rerank Score 5.8927 | Original FAISS Rank 1 | Paper paper2 | Chunk 49
Rank 2 | Rerank Score 5.8774 | Original FAISS Rank 8 | Paper paper2 | Chunk 87
Rank 3 | Rerank Score 5.5531 | Original FAISS Rank 12 | Paper paper1 | Chunk 30
Rank 4 | Rerank Score 4.7860 | Original FAISS Rank 2 | Paper paper2 | Chunk 41
Rank 5 | Rerank Score 4.7786 | Original FAISS Rank 14 | Paper paper2 | Chunk 40
